In [1]:
from nanogpt.layers import MultiHeadAttention, Embedding, FeedForward, LayerNorm
from nanogpt import Transformer
from nanogpt import Tokenizer, Indicer
from nanogpt import SlidingWindow
from nanogpt.loss_functions import CrossEntropyLoss
from nanogpt import Optimizer
import numpy as np
import os

## Data Loading & Preprocessing

In [2]:
path_to_text = "data/input.txt"
with open(path_to_text, 'r') as f:
    text = f.read()

def preprocess_text(text):
    text = text.lower()
    punctuation = ['.', ',', '!', '?', ';', ':', '"', "'", '(', ')', '[', ']', '{', '}', '-', '_', '/', '\\']
    for char in punctuation:
        text = text.replace(char, ' ' + char + ' ')
    return text

processed_text = preprocess_text(text)
print("Processed text:", processed_text[:100])

Processed text: first citizen : 
before we proceed any further ,  hear me speak . 

all : 
speak ,  speak . 

first 


## Tokenization & Encoding

In [3]:
tokenizer = Tokenizer(vocab_size=10000)
token_map = tokenizer.sequentialize(processed_text)
indicer = Indicer(vocab_size=10000)
indicer.fit(list(token_map))
encoded_tokens = indicer.encode(list(token_map))
vocab_size = len(set(processed_text.split()))
print("Encoded tokens:", encoded_tokens[:10])
print("Decoded text:", indicer.decode(encoded_tokens[:10]))
print("Vocabulary size:", vocab_size)
print("Total tokens:", len(encoded_tokens))

Encoded tokens: [305, 8658, 3808, 6845, 1942, 1265, 0, 7222, 6003, 8631]
Decoded text: ['first', 'citizen', ':', 'before', 'we', 'proceed', 'salute', 'further', ',', 'hear']
Vocabulary size: 11463
Total tokens: 262922


In [4]:
split_index = int(len(token_map) * 0.8)
train_token_map = token_map[:split_index]
test_token_map = token_map[split_index:]
encoded_train_tokens = indicer.encode(list(train_token_map))
encoded_test_tokens = indicer.encode(list(test_token_map))
print("Train tokens:", len(train_token_map))
print("Test tokens:", len(test_token_map))

Train tokens: 210337
Test tokens: 52585


## Sanity Checks

In [5]:
sliding_window = SlidingWindow(train_token_map)
sample = sliding_window.sample()
encoded_sample = indicer.encode(list(sample))
print("Sample shape:", np.array(encoded_sample).shape)

Sample shape: (512,)


In [6]:
embedding = Embedding(vocab_size, 768)
embedding.forward(encoded_sample).data

array([[-0.00115841,  0.00537668,  0.00656207, ..., -0.00882486,
        -0.01829642, -0.02147915],
       [-0.011392  ,  0.00481841,  0.00433596, ...,  0.00301173,
        -0.01156901, -0.00048682],
       [ 0.01738732,  0.01421156, -0.00994599, ...,  0.006839  ,
         0.00059693,  0.02156666],
       ...,
       [-0.01203616,  0.0127274 ,  0.00678808, ...,  0.01441877,
        -0.01926588,  0.00588851],
       [-0.01424921,  0.00704831,  0.00339618, ..., -0.01643641,
        -0.01089357,  0.00118093],
       [ 0.02408112, -0.02201984, -0.01662138, ...,  0.01013415,
        -0.02131124,  0.00370322]], shape=(512, 768))

In [7]:
transformer = Transformer(vocab_size, 768, 4, 3072, 4)
output = transformer.forward(encoded_sample)
print("Output shape:", output.data.shape)
print("Embedding weights shape:", transformer.embedding.weights.data.shape)
print("Projection weights shape:", transformer.projection.weights.data.shape)

Output shape: (512, 11463)
Embedding weights shape: (11463, 768)
Projection weights shape: (768, 11463)


## Training

In [8]:
def get_batch(data, block_size=256):
    i = np.random.randint(0, len(data) - block_size - 1)
    x = data[i : i + block_size]
    y = data[i + 1 : i + block_size + 1]
    return x, y

def visit(node, visited=None):
    """Check for vanishing gradients in the computation graph."""
    if visited is None:
        visited = set()
    if id(node) in visited:
        return
    visited.add(id(node))
    for child in node._children:
        visit(child, visited)
        if not np.any(child.grad):
            print("VANISHING GRADIENT FOUND")
            return

In [9]:
def train_loop(model, data, iters=100, lr=0.001):
    loss_func = CrossEntropyLoss()
    optimizer = Optimizer(loss_func, model)
    for i in range(iters):
        optimizer.zero_grad()
        x, y = get_batch(data)
        output = model.forward(x)
        loss = loss_func.forward(output, y)
        loss.backward()
        optimizer.step(lr)
        visit(loss)
        print(f"Iteration {i}: loss={loss.data:.4f}")

In [ ]:
model = Transformer(vocab_size, 768, 2, 3072, 2)
train_loop(model, encoded_train_tokens, iters=100)

Iteration 0: loss=9.4199
Iteration 1: loss=9.3993
Iteration 2: loss=9.3894
Iteration 3: loss=9.4052
Iteration 4: loss=9.3932
Iteration 5: loss=9.3811
Iteration 6: loss=9.3482
Iteration 7: loss=9.3741
Iteration 8: loss=9.3435
Iteration 9: loss=9.3143


## Text Generation

In [11]:
def generate(model, tokens, indicer, max_tokens=20, temperature=0.7):
    encoded_input = indicer.encode(list(tokens))
    text = list(encoded_input)
    for _ in range(max_tokens):
        output = model.forward(text).data
        logits = output[-1]
        scaled_logits = logits / temperature
        scaled_logits -= scaled_logits.max()  # numerical stability
        exps = np.exp(scaled_logits)
        probabilities = exps / np.sum(exps)
        token = np.random.choice(len(probabilities), p=probabilities)
        text.append(token)
    return text

In [12]:
prompt = "to be or not to be"
processed_prompt = preprocess_text(prompt)
processed_prompt = tokenizer.sequentialize(processed_prompt)

generated_tokens = generate(model, processed_prompt, indicer)
decoded_text = indicer.decode(generated_tokens)
print(" ".join(decoded_text))

to salute or salute to salute suggestion <unk> bottle <unk> reprieves thicket forenoon kitchens run bandy <unk> advertised obstacles opposite ally latches alban lesson doers travell
